In [1]:
import requests
import json
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv

load_dotenv('secrets.env')

True

In [3]:
# Google Authentication
PROJECT_ID = os.getenv('PROJECT_ID')
DATASET_ID = os.getenv('DATASET_ID')
TABLE_ID = 'vend_inventory'

# Replace with the path to your service account key file
SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')


In [5]:
# Initialize BigQuery client
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

# Define the BigQuery table reference
dataset_ref = client.dataset(DATASET_ID)
table_ref = dataset_ref.table(TABLE_ID)

In [6]:
schema = [
    bigquery.SchemaField("id", "STRING"),
    bigquery.SchemaField("outlet_id", "STRING"),
    bigquery.SchemaField("product_id", "STRING"),
    bigquery.SchemaField("inventory_level", "FLOAT"),
    bigquery.SchemaField("current_amount", "FLOAT"),
    bigquery.SchemaField("version", "INTEGER"),
    bigquery.SchemaField("deleted_at", "TIMESTAMP"),
    bigquery.SchemaField("average_cost", "FLOAT"),
    bigquery.SchemaField("reorder_point", "FLOAT"),
    bigquery.SchemaField("reorder_amount", "FLOAT")
]

In [7]:
# Create the table if it doesn't exist
table = bigquery.Table(table_ref, schema=schema)
table = client.create_table(table, exists_ok=True)

In [12]:
# Function to fetch data from Vend API
def fetch_vend_data(url, headers, params=None):
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()


In [13]:
# Function to load data into BigQuery
def load_data_to_bigquery(client, table_ref, rows_to_insert):
    errors = client.insert_rows_json(table_ref, rows_to_insert)
    if errors:
        print(f"Encountered errors while inserting rows: {errors}")
    else:
        print("Data successfully inserted into BigQuery.")


In [14]:
# Vend API details
vend_url = "https://ashcorp.retail.lightspeed.app/api/2.0/inventory"
vend_headers = {
    "accept": "application/json",
    "authorization": f"Bearer {os.getenv('LIGHTSPEED_ACCESS_TOKEN')}"
}

In [ ]:
# Pagination parameters
after = 0

# Extract, Transform, Load (ETL) process
while True:
    # Extract data from Vend API
    params = {"after": after}
    data = fetch_vend_data(vend_url, vend_headers, params)

    # Transform data (if needed)
    rows_to_insert = data["data"]
    
    # Load data into BigQuery
    load_data_to_bigquery(client, table_ref, rows_to_insert)
    
    # Get the max version number for the next request
    after = data["version"]["max"]

    
    # Check if the data collection is empty
    if not data["data"]:
        break

print("ETL process completed successfully.")